# ShotOptix Collaborative Ensemble (Google Colab GPU)

Train **XGBoost + LightGBM + CatBoost + Deep MLP** together, then stack their
probabilities to push shot-make accuracy toward **70–75%**.

## Setup
1. Runtime → Change runtime type → **GPU**
2. Upload/copy the project to Google Drive under `MyDrive/shot-optimization-engine`
3. Ensure `data/processed/shotoptix_ml_training.csv` is available
4. Run all cells

After training, download `colab_ensemble_bundle.joblib` and place it in
`backend/trained_models/` if you want to use it locally.

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install torch scikit-learn pandas numpy xgboost lightgbm catboost joblib
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import sys
import json
import time

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

ROOT_DIR = Path('/content/drive/MyDrive/shot-optimization-engine') if IN_COLAB else Path('..')
sys.path.insert(0, str(ROOT_DIR / 'backend'))
from app.ml.feature_builder import MODEL_FEATURES, build_features_from_dataframe, compute_prior_rates

DATA_PATH = ROOT_DIR / 'data/processed/shotoptix_ml_training.csv'
OUT_DIR = ROOT_DIR / 'backend/trained_models'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 2_000_000   # raise to None for all action-rich rows
REQUIRE_ACTION_TYPE = True
RANDOM_STATE = 42
N_FOLDS = 3
EPOCHS = 25
BATCH_SIZE = 8192

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
if REQUIRE_ACTION_TYPE:
    df = df[df['action_type'].fillna('').astype(str).str.len() > 0].copy()

if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df, _ = train_test_split(
        df, train_size=SAMPLE_SIZE, random_state=RANDOM_STATE, stratify=df['shot_made']
    )

y = df['shot_made'].astype(int)
train_df, test_df, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
prior_rates = compute_prior_rates(train_df)
X_train = build_features_from_dataframe(train_df, prior_rates=prior_rates).astype(np.float32)
X_test = build_features_from_dataframe(test_df, prior_rates=prior_rates).astype(np.float32)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print(f'Rows={len(df):,} train={len(X_train):,} test={len(X_test):,} feats={X_train.shape[1]} make={y.mean():.4f}')

In [ ]:
def metrics(y_true, probs, thr=0.5):
    pred = (probs >= thr).astype(int)
    return {
        'accuracy': float(accuracy_score(y_true, pred)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, probs)),
        'threshold': float(thr),
    }

def best_threshold(y_true, probs):
    best_thr, best = 0.5, metrics(y_true, probs, 0.5)
    for thr in np.arange(0.35, 0.66, 0.01):
        m = metrics(y_true, probs, thr)
        if m['accuracy'] > best['accuracy']:
            best, best_thr = m, float(thr)
    return best_thr, best

def pos_proba(model, X):
    p = model.predict_proba(X)
    classes = list(getattr(model, 'classes_', [0, 1]))
    idx = classes.index(1) if 1 in classes else 1
    return p[:, idx]

## 1) Tree models (collaborators)

In [ ]:
tree_models = {
    'xgboost': XGBClassifier(
        n_estimators=500, max_depth=7, learning_rate=0.04, subsample=0.85,
        colsample_bytree=0.8, min_child_weight=3, gamma=0.1, reg_alpha=0.1,
        reg_lambda=2.0, scale_pos_weight=spw, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1,
        tree_method='hist', device='cuda' if torch.cuda.is_available() else 'cpu',
    ),
    'lightgbm': LGBMClassifier(
        n_estimators=600, max_depth=8, learning_rate=0.04, num_leaves=127,
        subsample=0.85, colsample_bytree=0.8, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    ),
    'catboost': CatBoostClassifier(
        iterations=500, depth=7, learning_rate=0.04, l2_leaf_reg=5.0,
        auto_class_weights='Balanced', random_state=RANDOM_STATE, verbose=0,
        task_type='GPU' if torch.cuda.is_available() else 'CPU',
    ),
}

fitted = {}
holdout = {}
for name, model in tree_models.items():
    t0 = time.time()
    print(f'Training {name}...')
    try:
        model.fit(X_train, y_train)
    except Exception as exc:
        # CatBoost GPU can fail on some Colab images; fall back to CPU.
        print(f'  fallback CPU for {name}: {exc}')
        if name == 'catboost':
            model = CatBoostClassifier(
                iterations=500, depth=7, learning_rate=0.04, l2_leaf_reg=5.0,
                auto_class_weights='Balanced', random_state=RANDOM_STATE, verbose=0,
            )
            model.fit(X_train, y_train)
        elif name == 'xgboost':
            model.set_params(device='cpu')
            model.fit(X_train, y_train)
        else:
            raise
    fitted[name] = model
    holdout[name] = pos_proba(model, X_test)
    thr, m = best_threshold(y_test, holdout[name])
    print(f'  {name}: acc={m["accuracy"]:.4f} auc={m["roc_auc"]:.4f} thr={thr:.2f} ({time.time()-t0:.1f}s)')

## 2) Deep MLP collaborator (GPU)

In [ ]:
class ShotMLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 768),
            nn.BatchNorm1d(768),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(768, 384),
            nn.BatchNorm1d(384),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(384, 192),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(192, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Standardize for neural net stability
mean = X_train.mean(axis=0).values
std = X_train.std(axis=0).replace(0, 1).values
Xtr = ((X_train.values - mean) / std).astype(np.float32)
Xte = ((X_test.values - mean) / std).astype(np.float32)

mlp = ShotMLP(X_train.shape[1]).to(device)
opt = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
crit = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([spw], device=device)
)
loader = DataLoader(
    TensorDataset(torch.tensor(Xtr), torch.tensor(y_train.values.astype(np.float32))),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
Xte_t = torch.tensor(Xte).to(device)

best_state, best_auc = None, -1.0
for epoch in range(1, EPOCHS + 1):
    mlp.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(mlp(xb), yb)
        loss.backward()
        opt.step()
        total += float(loss.item())
    sched.step()
    mlp.eval()
    with torch.no_grad():
        probs = torch.sigmoid(mlp(Xte_t)).cpu().numpy()
    auc = roc_auc_score(y_test, probs)
    if auc > best_auc:
        best_auc = auc
        best_state = {k: v.detach().cpu().clone() for k, v in mlp.state_dict().items()}
    if epoch % 5 == 0 or epoch == 1:
        thr, m = best_threshold(y_test, probs)
        print(f'Epoch {epoch:02d} loss={total/len(loader):.4f} acc={m["accuracy"]:.4f} auc={auc:.4f}')

mlp.load_state_dict(best_state)
mlp.eval()
with torch.no_grad():
    holdout['mlp'] = torch.sigmoid(mlp(Xte_t)).cpu().numpy()
thr, m = best_threshold(y_test, holdout['mlp'])
print(f'mlp best: acc={m["accuracy"]:.4f} auc={m["roc_auc"]:.4f} thr={thr:.2f}')

## 3) Collaborate: soft average + stacking meta-learner

In [ ]:
holdout_df = pd.DataFrame(holdout)

# Soft average collaboration
avg_probs = holdout_df.mean(axis=1).to_numpy()
avg_thr, avg_m = best_threshold(y_test, avg_probs)
print('soft_average:', avg_m)

# Stacking on a validation slice of holdout (quick meta fit from OOF-like split of train)
# For Colab speed: fit meta on a validation split from train predictions of already-fit models
# Better: rebuild OOF. Here we use a secondary holdout from train for meta.
X_tr2, X_meta, y_tr2, y_meta = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)

meta_train = {}
for name, model in fitted.items():
    # Refit trees on X_tr2 for cleaner meta features
    from sklearn.base import clone
    m2 = clone(model)
    # XGBoost clone may keep device settings
    try:
        m2.fit(X_tr2, y_tr2)
    except Exception:
        if name == 'xgboost':
            m2.set_params(device='cpu')
        if name == 'catboost':
            m2 = CatBoostClassifier(
                iterations=400, depth=7, learning_rate=0.04, l2_leaf_reg=5.0,
                auto_class_weights='Balanced', random_state=RANDOM_STATE, verbose=0,
            )
        m2.fit(X_tr2, y_tr2)
    meta_train[name] = pos_proba(m2, X_meta)

# MLP meta features from current network on standardized X_meta
Xmeta_s = ((X_meta.values - mean) / std).astype(np.float32)
with torch.no_grad():
    meta_train['mlp'] = torch.sigmoid(mlp(torch.tensor(Xmeta_s).to(device))).cpu().numpy()

meta_X = pd.DataFrame(meta_train)
meta = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
meta.fit(meta_X, y_meta)

stack_probs = meta.predict_proba(holdout_df[meta_X.columns])[:, 1]
stack_thr, stack_m = best_threshold(y_test, stack_probs)
print('stacking:', stack_m)

summary = {
    'xgboost': best_threshold(y_test, holdout['xgboost'])[1],
    'lightgbm': best_threshold(y_test, holdout['lightgbm'])[1],
    'catboost': best_threshold(y_test, holdout['catboost'])[1],
    'mlp': best_threshold(y_test, holdout['mlp'])[1],
    'soft_average': avg_m,
    'stacking': stack_m,
}
print('\n=== Collaborative results ===')
for k, v in sorted(summary.items(), key=lambda kv: -kv[1]['accuracy']):
    print(f"{k:14s} acc={v['accuracy']:.4f} auc={v['roc_auc']:.4f} thr={v['threshold']:.2f}")

best_name = max(summary, key=lambda k: summary[k]['accuracy'])
print('\nBest:', best_name, summary[best_name]['accuracy'])
print('Target 70%:', 'REACHED' if summary[best_name]['accuracy'] >= 0.70 else 'NOT REACHED')

In [ ]:
bundle = {
    'base_models': fitted,
    'meta_model': meta,
    'model_order': list(meta_X.columns),
    'prior_rates': prior_rates,
    'features_used': list(MODEL_FEATURES),
    'mlp_state_dict': best_state,
    'mlp_mean': mean.tolist(),
    'mlp_std': std.tolist(),
    'best_method': best_name,
    'decision_threshold': summary[best_name]['threshold'],
    'results': summary,
}
out_path = OUT_DIR / 'colab_ensemble_bundle.joblib'
joblib.dump(bundle, out_path)
(OUT_DIR / 'colab_ensemble_results.json').write_text(json.dumps({
    'best_method': best_name,
    'results': summary,
    'rows': int(len(df)),
    'features': len(MODEL_FEATURES),
}, indent=2), encoding='utf-8')
print('Saved', out_path)
print('Download this file from Drive and keep it under backend/trained_models/')

if IN_COLAB:
    from google.colab import files
    files.download(str(out_path))